In [1]:
#makemore翻篇了！接下来是nanoGPT
#我们要正式开始进入Transformer了！

In [2]:
#!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
#这一步是下载数据集：tiny莎士比亚
#这一步在国内通常有些慢
#国内的读者可以用魔法直接打开网页后保存到目录里

In [3]:
#标准的解析文本写法，以只读方式用UTF-8编码打开文件为f,text就是超长的一个字符串 
with open('input.txt','r',encoding='utf-8') as f:
    text=f.read()

In [4]:
print(text[:1000])
print("\n总字符量:",len(text))

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [5]:
chars=sorted(list(set(text)))#这是我们用过好多次的提取字符的代码
vocab_size=len(chars)
for c in chars:
    print(repr(c),end="   ") # repr会把不可见字符转成\n这种可见写法
print("\n总字符量:",vocab_size)

'\n'   ' '   '!'   '$'   '&'   "'"   ','   '-'   '.'   '3'   ':'   ';'   '?'   'A'   'B'   'C'   'D'   'E'   'F'   'G'   'H'   'I'   'J'   'K'   'L'   'M'   'N'   'O'   'P'   'Q'   'R'   'S'   'T'   'U'   'V'   'W'   'X'   'Y'   'Z'   'a'   'b'   'c'   'd'   'e'   'f'   'g'   'h'   'i'   'j'   'k'   'l'   'm'   'n'   'o'   'p'   'q'   'r'   's'   't'   'u'   'v'   'w'   'x'   'y'   'z'   
总字符量: 65


In [6]:
stoi={ch:i for i,ch in enumerate(chars)}
itos={i:ch for i,ch in enumerate(chars)}
#还是构建字符表，但是现在我们可以构建函数来实现映射
encode=lambda s: [stoi[c] for c in s]
decode=lambda l: ''.join([itos[i] for i in l])

print(encode("Hello GPT!"))
print(decode(encode("Hello GPT!")))

[20, 43, 50, 50, 53, 1, 19, 28, 32, 2]
Hello GPT!


In [7]:
#我们用已经很熟悉的torch来包装text
import torch
data=torch.tensor(encode(text),dtype=torch.long)
print(data.shape)
print(data[:1000])

torch.Size([1115394])
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
        47, 59, 57

In [8]:
#区分训练集和验证集
n=int(0.9*len(text))
train_data=data[:n]
val_data=data[n+1:]

In [9]:
#接下来区分块
block_size=8
train_data[:block_size]

tensor([18, 47, 56, 57, 58,  1, 15, 47])

In [10]:
#在这里，我们会实现用前1～8个字符对应下一个字符
x=train_data[:block_size]
y=train_data[1:block_size+1]
for t in range(block_size):
    context=x[:t+1]
    target=y[t]
    print(f"context:{context},target:{target}")

context:tensor([18]),target:47
context:tensor([18, 47]),target:56
context:tensor([18, 47, 56]),target:57
context:tensor([18, 47, 56, 57]),target:58
context:tensor([18, 47, 56, 57, 58]),target:1
context:tensor([18, 47, 56, 57, 58,  1]),target:15
context:tensor([18, 47, 56, 57, 58,  1, 15]),target:47
context:tensor([18, 47, 56, 57, 58,  1, 15, 47]),target:58


In [11]:
#这里和之前的不一样了
#因为Transformer和之前的模型有一个不同点就是它不要求上下文是固定长度
#所以我们现在是可以用1～8个字符来预测下一个的

In [12]:
torch.manual_seed(1337)
batch_size=4#批大小
block_size=8#字符块大小


def get_batch(split):
    data=train_data if split=="train" else val_data
    ix=torch.randint(len(data)-block_size,(batch_size,))
    #torch.stack会用新维度把这些东西拼在一起
    #所以也就是对于x和y把4个形状为(8)的tensor拼成(4*8)的
    x=torch.stack([data[i:i+block_size]for i in ix])
    y=torch.stack([data[i+1:i+block_size+1]for i in ix])
    return x,y

#这里是展示我们的input和target
xb,yb=get_batch("train")
print("inputs:")
print(xb.shape)
print(xb)
print("target:")
print(yb.shape)
print(yb)

print('-----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
target:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
-----
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
when input is [44, 53

In [13]:
import torch
import torch.nn as nn#这个是torch的神经网络模块，我们之前手动实现的Linear等等都可以用这个实现
from torch.nn import functional as F
torch.manual_seed(1337)

#有的读者一开始看这里可能会不理解，其实我们就是开始用torch来代替我们的手写部分了
#还记得我们之前实现的embedding吗

# class Embedding:
#   def __init__(self, num_embeddings, embedding_dim):
#     self.weight = torch.randn((num_embeddings, embedding_dim))
#   def __call__(self, IX):
#     self.out = self.weight[IX]
#     return self.out
#   def parameters(self):
#     return [self.weight]

#这里几乎差不多
#构造就是一个vocab_size*vocab_size的矩阵
#调用就是一个矩阵乘法
#这里我们实现的是一个bigram
#所以我们直接用Embedding来当做我们的概率矩阵
#embedding就是个查表，所以把xb的全部元素都查了一下
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):#可以没有targets这样的话就可以直接用于生成
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets == None:
            loss=None
        else:
            #这几步是因为在pytorch内部，是按照BCT的顺序来排列的而不是BTC
            #所以我们需要调整一下
            #把X和Y压缩成一维的
            #这样就让BT全部压缩到B就符合条件了
            B,T,C=logits.shape
            logits=logits.view(B*T,C)
            targets=targets.view(-1)
            loss=F.cross_entropy(logits,targets)
            
        return logits,loss

    def generate(self, idx, max_new_tokens):#生成文字函数
        #这一步其实就是拿模型在生成新的字符
        #但是这次的生成方式是拼接
        #我们把新的字符直接在idx后面去拼接
        #这就非常像Transformer了
        #我们之前的生成的姓名是直接生成
        #现在是生成一个字之后再生成，再生成，全部拼成完整的一句话
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx
 

m = BigramLanguageModel(vocab_size)
logits,loss = m(xb, yb)#这一步是nn.Module的特性，相当于 m.forward(xb,yb)
#这一步就相当于xb去调用一下embedding
#embedding就是个查表，所以把xb的全部元素都查了一下
print(xb.shape)
print(yb.shape)
print(logits.shape)
print(loss)
#这里可以用0开始，去生成一串字符
print(decode(m.generate(torch.zeros((1,1),dtype=torch.long),max_new_tokens=100)[0].tolist()))

torch.Size([4, 8])
torch.Size([4, 8])
torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [14]:
optimizer=torch.optim.AdamW(m.parameters(),lr=1e-3)
#这是优化器，以后可以直接用这个来帮我们更新梯度，清空梯度了
#而且我们还可以用pytorch里各种各样的优化算法

In [15]:
batch_size=32
for steps in range(100):
    xb,yb=get_batch('train')
    logits,loss=m(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    print(loss.item())

4.692410945892334
4.664144515991211
4.765714645385742
4.70655632019043
4.5956573486328125
4.7101240158081055
4.713661193847656
4.686909198760986
4.700076103210449
4.718283653259277
4.715603351593018
4.684308052062988
4.745601177215576
4.735717296600342
4.666238784790039
4.58615255355835
4.714625835418701
4.671982765197754
4.715047359466553
4.74489164352417
4.630162715911865
4.707578182220459
4.670665740966797
4.582583427429199
4.739546298980713
4.674807071685791
4.805595874786377
4.749917507171631
4.691989421844482
4.604404926300049
4.721841335296631
4.741591930389404
4.609963417053223
4.662769794464111
4.730099678039551
4.738433361053467
4.688235282897949
4.639987945556641
4.736632823944092
4.709773540496826
4.736939430236816
4.69184684753418
4.719646453857422
4.752516746520996
4.570086479187012
4.643786907196045
4.699163913726807
4.806960105895996
4.572142601013184
4.717066287994385
4.509502410888672
4.603540897369385
4.6649675369262695
4.712099075317383
4.736577033996582
4.812878131

In [16]:
#有优化，但不多
batch_size=32
for steps in range(10000):
    xb,yb=get_batch('train')
    logits,loss=m(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if steps%1000==0:
        print(loss.item())
print(loss.item())

4.621085166931152
3.6824679374694824
2.997836112976074
2.7563703060150146
2.7407634258270264
2.4976727962493896
2.5222768783569336
2.465423583984375
2.5472488403320312
2.5384695529937744
2.451270580291748


In [17]:
#基本下降到我们之前所说的bigram模型的理论最低值了

In [18]:
print(decode(m.generate(torch.zeros((1,1),dtype=torch.long),max_new_tokens=500)[0].tolist()))


ARICI fere yothar highe LUSeppe y chan d a t, hif nes,
Pl peORjulaliseel E Lo sot.
Ouprco,
Areke atha anghars s NCHO:
Cieneedevee, ESons, s ffaye nencawhe;
UThis thoo lfantentROR: scor ce athecre yealldil:
Sourorir Goban o ishead onshatie cth walot hery I h merd ply this.
TAne bexpimond me stowan ureimy herst P,
N s, orowha whet brtore hilave w wese wedortore 's r Re, nduisstive ly s; bl! br'sithoumat n a tistheseis dy hrme ho od, ar
I al t!
A hen heand caousth om hs havean bld; t Thace t?

T:
A


In [19]:
#可以看到，虽然在胡言乱语，但至少长得有点像了哈哈哈哈

In [20]:
#接下来我们就要开始引入注意力机制了

In [21]:
torch.manual_seed(1337)
B,T,C=4,8,2
x=torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [22]:
#在这里，我们一个batch里有8个token
#注意力机制希望他们可以互相通讯
#首先最基本的是：因为我们的GPT是自回归语言模型，所以有个基本规则是我们的通讯只能向前通讯，因为我们不能预测未来
#然后我们需要让每一次预测能够得到之前的信息
#那怎么整合前文的信息呢
#最简单的方式就是直接把前文的激活值求平均传下去
#虽然这样很简陋，信息压缩率也非常大但是确实是一种上下文机制
#也就是说：我们对于前面的token每一个都平等地关注一下

In [23]:
xbow = torch.zeros((B,T,C))#bow是bag of words的缩写，这是对于内容进行平均处理的俗语
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1] #取x[b,0]~x[b,t],最后的形状是(t,C)
        xbow[b,t] = torch.mean(xprev, 0)#xbow[b,t]形状是(C),在0维度求平均的结果也是(C)
print(xbow[b,t].shape)
print(xprev.shape)

torch.Size([2])
torch.Size([8, 2])


In [24]:
x[0],xbow[0]

(tensor([[ 0.1808, -0.0700],
         [-0.3596, -0.9152],
         [ 0.6258,  0.0255],
         [ 0.9545,  0.0643],
         [ 0.3612,  1.1679],
         [-1.3499, -0.5102],
         [ 0.2360, -0.2398],
         [-0.9211,  1.5433]]),
 tensor([[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]]))

In [25]:
#读者可以仔细观察一下这两个
#可以发现xbow的每一行都是x的从第一行到这一行的求平均

In [26]:
#其实这个过程可以被矩阵乘法给加速
torch.manual_seed(42)
a=torch.ones((3,3))
b=torch.randint(0,10,(3,2)).float()
c=a@b
print("a:",a)
print("b:",b)
print("c:",c)

a: tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]])
b: tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
c: tensor([[14., 16.],
        [14., 16.],
        [14., 16.]])


In [27]:
#我们会发现c就是b的每一列的求和

In [28]:
#torch还实现了三角矩阵
torch.tril(torch.ones((3,3)))

tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])

In [29]:
#再试试看
torch.manual_seed(42)
a=torch.tril(torch.ones((3,3)))
b=torch.randint(0,10,(3,2)).float()
c=a@b
print("a:",a)
print("b:",b)
print("c:",c)

a: tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])
b: tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
c: tensor([[ 2.,  7.],
        [ 8., 11.],
        [14., 16.]])


In [30]:
#发现没有，直接就是我们的求和了
torch.manual_seed(42)
a=torch.tril(torch.ones((3,3)))
b=torch.randint(0,10,(3,2)).float()
a=a/a.sum(1,keepdim=True)
c=a@b
print("a:",a)
print("b:",b)
print("c:",c)

a: tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
b: tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
c: tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [31]:
#这就是我们要的求平均
#那怎么运用到我们之前的BTC呢
wei=torch.tril(torch.ones(T,T))
wei=wei/wei.sum(1,keepdim=True)

In [32]:
xbow2=wei@x
#这一步是(T,T)@(B,T,C)=?
#pytorch的广播机制会增加一个维度
#变成(B,T,T)@(B,T,C)
#然后对于每个B，进行(T,T)@(T,C)的操作
#最后的形状应该是(B,T,C)
print(xbow2.shape)

torch.Size([4, 8, 2])


In [33]:
#验证是对的
#而且这样看：
#对于x的每一个(T,C),都进行了一次求平均的计算
#这和我们之前的计算是一致的
torch.allclose(xbow,xbow2)

True

In [34]:
#两者是一样的

In [35]:
#这里还有第三种写法，读者自己领悟吧，这里的线性代数太巧妙了
# version 3: use Softmax
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))#这里是用tril来限制禁止看到未来的东西
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
torch.allclose(xbow, xbow3)

True

In [36]:
#好的！接下来我们可以实现self-attention了！
torch.manual_seed(1337)
B,T,C=4,8,32
x=torch.randn((B,T,C))

tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
out = wei @ x

out.shape

torch.Size([4, 8, 32])

In [37]:
#这个是刚刚的代码，我们对前文的每一个值都关注一下
#但是这不符合我们的逻辑
#比如对于：
#I have a girlfriend,she is beautiful
#这句话中，对于"she"这个单词，显然我们应该关注"girlfriend"这个词更多一些
#所以我们需要模型是可以训练自身的注意力的

In [38]:
torch.manual_seed(1337)
B,T,C=4,8,32
x=torch.randn((B,T,C))

head_size=16
key=nn.Linear(C,head_size,bias=False)
query=nn.Linear(C,head_size,bias=False)
k=key(x)#(4,8,16)
q=query(x)#(4,8,16)
wei=q @ k.transpose(-2,-1)#(4,8,16) @ (4,16,8) --> (4,8,8)

tril = torch.tril(torch.ones(T, T))
#wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
out = wei @ x
print('x:',x.shape)
print('wei:',wei.shape)
print("out:",out.shape)

x: torch.Size([4, 8, 32])
wei: torch.Size([4, 8, 8])
out: torch.Size([4, 8, 32])


In [39]:
#我们来理解一下这段
#wei是什么？wei是weight的缩写
#这个wei的形状是(4,8,8)
#我们去掉batch，就是(8,8)
#所以其实这是个表格：
#代表的是：这8个token之间相互的关联性
#组合成一张表就是(8,8)

In [40]:
#而要采用key和query的原因是：我们要学习的东西和字符本身是有关系的
#比如刚刚我们的例子
#I have a girlfriend,she is beautiful
#这里关联的是'girlfriend'和'it'的共同作用的结果
#所以其实我们关注的并不是位置本身，而是每个位置上的词
#所以我们需要key和query
#key:键值 query:查询
#每个词有自己的意义，同时也对别的词有特殊的关注度
#所以我们采用了这样的乘积方式，用词本身来构造这个表

In [41]:
wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)

In [42]:
torch.manual_seed(1337)
B,T,C=4,8,32
x=torch.randn((B,T,C))

head_size=16
key=nn.Linear(C,head_size,bias=False)
query=nn.Linear(C,head_size,bias=False)
value=nn.Linear(C,head_size,bias=False)
k=key(x)#(4,8,16)
q=query(x)#(4,8,16)
wei=q @ k.transpose(-2,-1)#(4,8,16) @ (4,16,8) --> (4,8,8)

tril = torch.tril(torch.ones(T, T))
#wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v=value(x)
out = wei @ v
print('x:',x.shape)
print('wei:',wei.shape)
print("out:",out.shape)

x: torch.Size([4, 8, 32])
wei: torch.Size([4, 8, 8])
out: torch.Size([4, 8, 16])


In [43]:
# 这里我们多引入了一个 V
# Q 和 K 主要负责决定“谁应该关注谁”
#
# 比如：
# "Harry gave me an apple, it is delicious."
# 对于 "it"：
# 我们用 Q 和 K 计算相关性，
# 让 "it" 对 "apple" 产生较高的 attention weight。

# 但是，找到 "apple" 之后，
# 我们还需要决定“从 apple 那里获取什么信息”。
# apple 这个词本身可能有很多含义：
# 可以指水果，也可以和 Apple 公司、苹果手机等概念有关。

# 这就是 V 发挥作用的地方：
# V = value(x)
# 它把 token 的原始表示进一步变换，
# 得到当前 Attention 中需要传递的信息。
# 在这个句子中，模型可能学会让与“水果”相关的信息
# 在 V 中更加有用，而其他不相关的信息影响较小。

# 最后：
# attention weights 决定“从谁那里拿多少信息”
# V 决定“拿到的这些信息是什么”

In [44]:
#这就是我们的"self-attention"
#之所以是self，是因为我们的QKV都是依赖x生产的

In [45]:
#并且显而易见：我们的自注意力机制部分是要使用softmax的，所以我们最好也要采用kaiming初始化的思路
wei=q @ k.transpose(-2,-1)*head_size**-0.5

tril = torch.tril(torch.ones(T, T))
#wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v=value(x)
out = wei @ v
print('x:',x.shape)
print('wei:',wei.shape)
print("out:",out.shape)

x: torch.Size([4, 8, 32])
wei: torch.Size([4, 8, 8])
out: torch.Size([4, 8, 16])


In [46]:
#这一串，就是所谓的self-attention head
class Head(nn.Module):

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)

        self.value = nn.Linear(n_embd, head_size, bias=False)

        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x) 
        q = self.query(x) 

        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 
        
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) 
        wei = F.softmax(wei, dim=-1) # (B, T, T)

        wei = self.dropout(wei)
        
        v = self.value(x)
        out = wei @ v 
        
        return out

In [48]:
#从这里开始andrej开始经常打开vscode去编辑他的v2.py代码了😵‍💫
#我在这里把他的代码摘下来了，但是很遗憾是完整版的，不能像notebook一样实现过程化
#我也添加了中文注释
#并且微调了参数。原版代码andrej在一块A100上跑了15分钟😱，调整了一下已经可以在mac上跑起来了，当然精确度差了很多
#欢迎读者来读一读

In [49]:
#nanoGPT第一部分就这么多了